# Solutions to L13

```{solution-start} l13-passive-cube
:class: dropdown
:label: sol-l13-passive_cube
See code below. 
```

In [ ]:
from dolfinx import fem, mesh, io, default_scalar_type
import dolfinx.fem.petsc
from ufl import (
    TestFunction,
    Measure,
    FacetNormal,
    variable,
    Identity,
    grad,
    diff,
    dot,
    inner,
    tr,
    det,
    inv,
    dx,
)
from mpi4py import MPI
from scifem import evaluate_function
from matplotlib import pyplot as plt
import numpy as np
from plotting import setup_gif_visualizer, update_gif_frame

# Create the mesh and the function space for the solutions
domain = mesh.create_unit_cube(MPI.COMM_WORLD, 4, 4, 4)
V = fem.functionspace(domain, ("Lagrange", 2, (domain.geometry.dim,)))

# Define functions
v = TestFunction(V)  # Test function
u = fem.Function(V, name="u")  # Displacement

# Mark boundary subdomains
fdim = domain.topology.dim - 1  # facet dimension


# Define marker functions for the left and right boundaries
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], 1.0)


## Locate the degrees of freedom and facets on the left and right boundaries
fdim = domain.topology.dim - 1
facets_l = mesh.locate_entities_boundary(domain, fdim, left)
dofs_l = fem.locate_dofs_topological(V, fdim, facets_l)

facets_r = mesh.locate_entities_boundary(domain, fdim, right)
dofs_r = fem.locate_dofs_topological(V, fdim, facets_r)

# Generate the meshtags, tagging the left and right boundaries with different markers
marker_l = 1
marker_r = 2

# Get ownership information regarding facets
facet_map = domain.topology.index_map(fdim)
num_facets = facet_map.size_local + facet_map.num_ghosts
facet_values = np.full(num_facets, -1, dtype=np.int32)
facet_values[facets_l] = marker_l
facet_values[facets_r] = marker_r
boundary_markers = mesh.meshtags(
    domain,
    fdim,
    np.arange(num_facets, dtype=np.int32),
    facet_values,
)

# Redefine boundary measure to allow integrals over part of boundary
ds = Measure("ds", domain, subdomain_data=boundary_markers)

# Define Dirichlet boundary condition on left boundary
bc = fem.dirichletbc(np.zeros(3, dtype=default_scalar_type), dofs_l, V)
bcs = [bc]

# Kinematics
d = len(u)
I = variable(Identity(d))  # Identity tensor
F = variable(I + grad(u))  # Deformation gradient
C = variable(F.T * F)  # Right Cauchy-Green tensor
E = variable(0.5 * (C - I))  # Green-Lagrange strain tensor

# Material parameters (Lamé parameters)
mu = 4.0
lmbda = 20.0

# The strain energy for the St-Venant Kirchhoff model:
psi = lmbda / 2 * (tr(E) ** 2) + mu * tr(E * E)

S = diff(psi, E)  # Second Piola-Kirchhoff stress
P = F * S  # First Piola-Kirchhoff stress (alt. P = diff(psi, F))

p_right = fem.Constant(domain, 0.0)  # the pressure load (zero for now)

# Definition of the weak form:
N = FacetNormal(domain)
traction = -p_right * det(F) * dot(inv(F).T, N)
# Residual: Internal forces - External forces
R = inner(grad(v), P) * dx - inner(v, traction) * ds(2)

# Set up nonlinear problem
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_error_if_not_converged": True,
    "ksp_error_if_not_converged": True,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
 #   "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="passive_cube_",
)

# Prepare output file
outfile = io.VTXWriter(domain.comm, "output/passive_cube_u.bp", [u])
outfile.write(0.0)
# Prepare GIF
plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/passive_cube.gif"
)

# Finally, we solve the problem for different loads, and plot the load vs displacement.

# Step-wise loading (for plotting and convergence)
load_steps = 5
target_load = 10.0
loads = np.linspace(0, target_load, load_steps)
disps = np.zeros(load_steps)
track_point = [1.0, 0.5, 0.5]

for step in range(load_steps):

    # Update traction value
    p_right.value = -loads[step]

    # Solve the nonlinear problem
    problem.solve()

    # Evaluate displacement at point defined above
    disps[step] = evaluate_function(u, [track_point])[0][0]

    # Write GIF frame
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

    # Write displacement to file
    outfile.write(loads[step])

outfile.close()
plotter.close()

plt.figure()
plt.plot(loads, disps, ".-")
plt.xlabel("Applied pressure load")
plt.ylabel(r"Displacement $u_x$ of point (1.0, 0.5, 0.5)")
plt.title("Passive cube stretching")
plt.show()

In [ ]:
# Display the generated GIF
from IPython.display import Image
Image(filename="output/passive_cube.gif", width=500)

```{solution-end}
```

```{solution-start} l13-active-cube
:class: dropdown
:label: sol-l13-active-cube
See code below. 
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dolfinx import fem, mesh, io, default_scalar_type
import dolfinx.fem.petsc
from ufl import (
    TestFunction,
    Measure,
    FacetNormal,
    variable,
    Identity,
    grad,
    diff,
    dot,
    inner,
    tr,
    det,
    inv,
    dx,
    as_vector,
)
from mpi4py import MPI
from scifem import evaluate_function

from guccionematerial import GuccioneMaterial
from plotting import setup_gif_visualizer, update_gif_frame

# Setup the mesh and the function space for the solutions
domain = mesh.create_unit_cube(MPI.COMM_WORLD, 4, 4, 4)
V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))

# Define functions
v = TestFunction(V)  # Test function
u = fem.Function(V)  # Displacement from previous iteration

# Mark boundary subdomains
fdim = domain.topology.dim - 1


# Define marker functions for the left and right boundaries
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], 1.0)


# Locate the degrees of freedom and facets on the left and right boundaries
fdim = domain.topology.dim - 1
facets_l = mesh.locate_entities_boundary(domain, fdim, left)
dofs_l = fem.locate_dofs_topological(V, fdim, facets_l)

facets_r = mesh.locate_entities_boundary(domain, fdim, right)
dofs_r = fem.locate_dofs_topological(V, fdim, facets_r)

# Generate the meshtags, tagging the left and right boundaries with different markers
marker_l = 1
marker_r = 2

# Get ownership information regarding facets
facet_map = domain.topology.index_map(fdim)
num_facets = facet_map.size_local + facet_map.num_ghosts
facet_values = np.full(num_facets, -1, dtype=np.int32)
facet_values[facets_l] = marker_l
facet_values[facets_r] = marker_r
boundary_markers = mesh.meshtags(
    domain,
    fdim,
    np.arange(num_facets, dtype=np.int32),
    facet_values,
)

# Redefine boundary measure
ds = Measure("ds", domain, subdomain_data=boundary_markers)

# Define Dirichlet boundary condition on left boundary
clamp = fem.Constant(domain, np.zeros(3, dtype=default_scalar_type))
bc = fem.dirichletbc(clamp, dofs_l, V)
bcs = [bc]

# Define a point (1.0, 0.5, 0.5) on the right boundary to track the displacement in
track_point = [1.0, 0.5, 0.5]

# Kinematics
d = len(u)
I = Identity(d)  # Identity tensor
F = I + grad(u)  # Deformation gradient
F = variable(F)
C = F.T * F  # the right Cauchy-Green tensor
E = 0.5 * (C - I)  # the Green-Lagrange strain tensor

# Material parameters (Lamé parameters)
mu = 4.0
lmbda = 20.0

# Tissue microstructure (needed by the GuccioneMaterial class)
f0 = as_vector([1.0, 0.0, 0.0])
s0 = as_vector([0.0, 1.0, 0.0])
n0 = as_vector([0.0, 0.0, 1.0])

# Create an instance of the material model class
material = GuccioneMaterial(domain, e1=f0, e2=s0, e3=n0, kappa=1e2, Tactive=0.0)

# Differentiate the strain energy to get stress
psi = material.strain_energy(F)
P = diff(psi, F)

# For this example we keep the pressure at zero, but keep it here for flexibility
p_right = fem.Constant(domain, 0.0)

# Definition of the weak form
N = FacetNormal(domain)
traction = p_right * det(F) * dot(inv(F).T, N)
R = inner(P, grad(v)) * dx - inner(v, traction) * ds(2)

# Set up nonlinear problem
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_error_if_not_converged": True,
    "ksp_error_if_not_converged": True,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
 #   "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="active_cube_",
)

# Prepare output file
outfile = io.VTXWriter(domain.comm, "output/active_cube_u.bp", [u])
outfile.write(0.0)
# Prepare GIF
plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/active_cube.gif", clim=[0, 0.2]
)

# Ramp up the active tension from 0 to 5 in 6 steps
active_steps = 6
target_active = 5.0
active = np.linspace(0, target_active, active_steps, dtype=np.float64)
disps = np.zeros(active_steps)


for step in range(active_steps):

    # Update active tension value
    material.set_active_stress(active[step])

    # Solve the nonlinear problem
    problem.solve()

    # Evaluate displacement at point defined above
    disps[step] = evaluate_function(u, [track_point])[0][0]  # extract x comp.

    # Write GIF frame
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

    # Write displacement to file
    outfile.write(step)

outfile.close()
plotter.close()

plt.figure()
plt.plot(disps, active)
plt.xlabel(r"Displacement $u_x$ of point (1.0, 0.5, 0.5)")
plt.ylabel("Applied pressure load")
plt.show()

In [ ]:
# Display the generated GIF
from IPython.display import Image
Image(filename="output/active_cube.gif", width=500)

```{solution-end}
```

```{solution-start} l13-ellipsoid
:class: dropdown
:label: sol-l13-ellipsoid
See code below. 
```

In [ ]:

from dolfinx import fem, io, plot, default_scalar_type
import dolfinx.fem.petsc
from ufl import (
    TestFunction,
    Measure,
    FacetNormal,
    variable,
    Identity,
    SpatialCoordinate,
    grad,
    diff,
    dot,
    inner,
    tr,
    det,
    inv,
    dx,
    as_vector,
)
from mpi4py import MPI
import io4dolfinx

import pyvista
import matplotlib.pyplot as plt
import numpy as np

from guccionematerial import GuccioneMaterial
from plotting import setup_gif_visualizer, update_gif_frame

def load_ellipsoid_data():
    """Returns 4-tuple:
    domain - the mesh,
    mf - MeshTags defining boundary markers,
    numbering - dict of marking numbers,
    fibers - list of functions defining microstructure
    """

    # Load the mesh and boundary markers (MeshTags) from XDMF
    with io.XDMFFile(MPI.COMM_WORLD, "lv-mesh/mesh.xdmf", "r") as xdmf:
        # Check your XDMF file to ensure the name="Mesh" matches
        domain = xdmf.read_mesh(name="Mesh")

        # In FEniCSx, we must compute connectivity before dealing with facets
        domain.topology.create_connectivity(
            domain.topology.dim - 1, domain.topology.dim
        )

        # Read the facet tags
        mf = xdmf.read_meshtags(domain, name="Facet tags")

    # Scale mesh from millimeters to centimeters
    domain.geometry.x[:] *= 0.1

    numbering = {"BASE": 5, "ENDO": 6, "EPI": 7}

    # Setup function space and functions for fibers
    fiber_space = fem.functionspace(domain, ("Lagrange", 2, (domain.geometry.dim,)))
    fiber = fem.Function(fiber_space, name="f0")
    sheet = fem.Function(fiber_space, name="s0")
    cross_sheet = fem.Function(fiber_space, name="n0")

    # Read the fiber directions from file using io4dolfinx
    bp_filepath = "lv-mesh/geometry.bp"
    io4dolfinx.read_function(bp_filepath, fiber)
    io4dolfinx.read_function(bp_filepath, sheet)
    io4dolfinx.read_function(bp_filepath, cross_sheet)

    fibers = [fiber, sheet, cross_sheet]

    return domain, mf, numbering, fibers

def compute_cavity_volume(mesh, mf, numbering, u=None):
    X = SpatialCoordinate(mesh)
    N = FacetNormal(mesh)

    if u is not None:
        I = Identity(3)
        F = I + grad(u)
        J = det(F)
        vol_form = (-1.0 / 3.0) * dot(X + u, J * inv(F).T * N)
    else:
        vol_form = (-1.0 / 3.0) * dot(X, N)

    ds = Measure("ds", domain=mesh, subdomain_data=mf)

    return fem.assemble_scalar(fem.form(vol_form * ds(numbering["ENDO"])))


domain, facet_tags, numbering, fibers = load_ellipsoid_data()

V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))

# Redefine boundary measure, to allow integration over parts of boundary
metadata = {"quadrature_degree": 4}
ds = Measure("ds", domain=domain, subdomain_data=facet_tags, metadata=metadata)
dx = Measure("dx", domain=domain, metadata=metadata)

clamp = np.zeros(domain.geometry.dim, dtype=default_scalar_type)
base_dofs = fem.locate_dofs_topological(
    V, facet_tags.dim, facet_tags.find(numbering["BASE"])
)
bc = fem.dirichletbc(clamp, base_dofs, V)
bcs = [bc]

# Define solution u and test function v
u = fem.Function(V)
v = TestFunction(V)

# Define strain measures
I = Identity(3)  # the identity matrix
F = I + grad(u)  # the deformation gradient
F = variable(F)

mat = GuccioneMaterial(
    domain, e1=fibers[0], e2=fibers[1], e3=fibers[2], kappa=1e3, Tactive=0.0
)
psi = mat.strain_energy(F)
P = diff(psi, F)  # the first Piola-Kirchoff stress tensor

p_endo = fem.Constant(domain, 0.0)

# Define nonlinear problem
N = FacetNormal(domain)
Gext = (
    p_endo * inner(v, det(F) * inv(F) * N) * ds(numbering["ENDO"])
)  # endocardial pressure
R = inner(P, grad(v)) * dx - Gext

# Step-wise loading (for plotting and convergence)
pressure_steps = 20
active_steps = 20
target_pressure = 10.0
target_active = 20.0

# first ramp up pressure, then keep constant
filling_pressure = np.linspace(0, target_pressure, pressure_steps)
const_pressure = np.ones(active_steps) * target_pressure
pressures = np.concatenate((filling_pressure, const_pressure))

# zero active tension during filling, then increase linearly
active1 = np.zeros_like(filling_pressure)
active2 = np.linspace(0, target_active, active_steps)
active = np.concatenate((active1, active2))

petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_error_if_not_converged": True,
    "ksp_error_if_not_converged": True,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
 #   "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="nonlinear_basic",
)

plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/ellipsoid_contraction.gif", clim=[0, 0.4]
)

filename = "output/ellipsoid_contraction.bp"
outfile = io.VTXWriter(domain.comm, filename, [u])
outfile.write(0.0)


volumes = np.zeros_like(pressures)
for i, step in enumerate(range(pressure_steps + active_steps)):
    p_endo.value = -pressures[step]
    mat.set_active_stress(active[step])
    
    problem.solve()
    volumes[step] = compute_cavity_volume(domain, facet_tags, numbering, u)
    
    outfile.write(i)
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

plotter.close()
outfile.close()

In [ ]:
plt.figure()
plt.plot(volumes, pressures)
plt.xlabel("Volume")
plt.ylabel("Pressure")
plt.show()

In [ ]:
# Display the generated GIF
from IPython.display import Image
Image(filename="output/ellipsoid_contraction.gif", width=500)

```{solution-end}
```